## Notebook — Sync Lucca → Supabase (BE Saisies de temps)

### Objectif
Alimenter la table Supabase `lucca_saisie_temps` (mirror) depuis la table Fabric
déjà nettoyée `Lakehouse_Gold.lucca_time_fusion_gold`.

### Source
- `Lakehouse_Gold.lucca_time_fusion_gold` filtrée sur `source_system = 'LUCCA'`
  (EURECIA legacy = historique, pas pris pour ce MVP — élargir si besoin)
- **Reconstitution code_affaire 8 chars** : `code` (5 chars) + `activite` (3 chars)
  - Ex: `EVINZ` + `ETD` = `EVINZETD` → match `be_affaires.code_affaire`
  - Ex: `EVINZ` + `000` = `EVINZ000` → temps "projet sans activité précise"
    (gardé en base, ne match pas une affaire spécifique mais visible
    dans des stats au niveau projet 5 chars)

### Destination
- `lucca_saisie_temps` — clé idempotence `external_id` = MD5 composite
  `(source_system | user_id | startsAt_date | code | activite)`
- Lookup `id_lucca → profile_id` (Supabase) au démarrage pour remplir `user_id`

### Pattern d'envoi
Edge Function `bulk-upsert` (mode `upsert` total). Conflict key = `external_id`.
Idempotent — relançable, le delta s'additionne sans doublons. Si une saisie
est corrigée côté Lucca → re-fusion gold → même hash → upsert (UPDATE).

In [ ]:
# ─────────────────────────────────────────
# 0) Credentials & helpers
# ─────────────────────────────────────────

# env = notebookutils.variableLibrary.getLibrary("env-lucca")
# SUPABASE_URL       = env.SUPABASE_URL
# SYNC_SECRET        = env.SYNC_SECRET
# SUPABASE_ANON_KEY  = env.SUPABASE_PUBLISHABLE_KEY

import math, time, re, requests
from decimal import Decimal
from datetime import date, datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, DateType, LongType

GOLD_TABLE      = "Lakehouse_Gold.lucca_time_fusion_gold"
BULK_UPSERT_URL = f"{SUPABASE_URL.rstrip('/')}/functions/v1/bulk-upsert"
BATCH_SIZE      = 500
SLEEP_S         = 0.02

SOURCE_FILTER = "LUCCA"

def supabase_headers(json_body: bool = True):
    h = {
        "Authorization":  f"Bearer {SUPABASE_ANON_KEY}",
        "apikey":         SUPABASE_ANON_KEY,
        "x-sync-secret":  SYNC_SECRET,
    }
    if json_body:
        h["Content-Type"] = "application/json"
    return h

_CTRL_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")

def _clean_str(s):
    if s is None: return None
    return _CTRL_RE.sub("", str(s)).rstrip() or None

def _to_jsonable(v):
    if v is None: return None
    if isinstance(v, bool): return v
    if isinstance(v, int): return v
    if isinstance(v, float):
        return None if (math.isnan(v) or math.isinf(v)) else round(v, 4)
    if isinstance(v, Decimal):
        try:
            f = float(v); return None if (math.isnan(f) or math.isinf(f)) else round(f, 4)
        except Exception:
            return _clean_str(str(v))
    if isinstance(v, (date, datetime)): return v.isoformat()
    if isinstance(v, str):
        s = _clean_str(v)
        if not s or s in ("1899-12-30", "0000-00-00"): return None
        return s
    if isinstance(v, dict): return {k: _to_jsonable(val) for k, val in v.items()}
    if isinstance(v, (list, tuple)): return [_to_jsonable(x) for x in v]
    return _clean_str(str(v))

def _sanitize(rec):
    return {k: _to_jsonable(v) for k, v in rec.items()}

def _post_batch(table: str, conflict_key: str, records: list):
    if not records: return 0, None
    payload = {
        "table":        table,
        "conflict_key": conflict_key,
        "records":      [_sanitize(r) for r in records],
        "on_conflict":  "upsert",
    }
    try:
        r = requests.post(BULK_UPSERT_URL, headers=supabase_headers(), json=payload, timeout=180)
        if r.status_code in (200, 201, 204):
            return len(records), None
        return 0, f"HTTP {r.status_code}: {r.text[:300]}"
    except Exception as e:
        return 0, str(e)

def send_to_supabase(table: str, conflict_key: str, records: list, label: str):
    ok_total, err_total, err_msgs = 0, 0, []
    total = len(records)
    for i in range(0, total, BATCH_SIZE):
        chunk = records[i:i + BATCH_SIZE]
        nb_ok, err = _post_batch(table, conflict_key, chunk)
        ok_total += nb_ok
        if err:
            err_total += len(chunk); err_msgs.append(f"batch {i//BATCH_SIZE + 1}: {err}")
        time.sleep(SLEEP_S)
    print(f"[{label}] ✅ {ok_total}/{total} envoyés | ❌ {err_total} erreurs")
    for m in err_msgs: print(f"  ⚠️  {m}")
    return ok_total, err_total

print("[0] Config OK")
print(f"    Source table  : {GOLD_TABLE}")
print(f"    Source filter : source_system = '{SOURCE_FILTER}'")

In [ ]:
# ─────────────────────────────────────────
# 1) Lookup id_lucca → profile_id (depuis Supabase)
# ─────────────────────────────────────────

url = f"{SUPABASE_URL.rstrip('/')}/rest/v1/profiles"
params = {
    "select":   "id,id_lucca",
    "id_lucca": "not.is.null",
    "limit":    "5000",
}
r = requests.get(url, headers=supabase_headers(json_body=False), params=params, timeout=60)
r.raise_for_status()
profiles_rows = r.json()

ID_LUCCA_TO_PROFILE_ID = {}
for row in profiles_rows:
    if row.get("id_lucca"):
        try:
            key = int(str(row["id_lucca"]).strip())
            ID_LUCCA_TO_PROFILE_ID[key] = row["id"]
        except Exception:
            continue

print(f"[1] Lookup map : {len(ID_LUCCA_TO_PROFILE_ID)} profils Lucca -> Supabase")

In [ ]:
# ─────────────────────────────────────────
# 2) Lecture Spark + filtres + reconstitution code_site (8 chars) + external_id
# ─────────────────────────────────────────

df = spark.table(GOLD_TABLE)
print(f"[2] {GOLD_TABLE} chargé : {df.count()} lignes brutes")

# Filtres :
#  - source_system = LUCCA
#  - code (5 chars) renseigné
#  - activite (3 chars) renseignée (y compris '000' = temps projet sans activité précise,
#    garde en base, ne match pas une affaire mais utile pour stats projet)
#  - user_id renseigné, date renseignée, durée > 0
df_filtered = (
    df
    .filter(F.col("source_system") == SOURCE_FILTER if SOURCE_FILTER else F.lit(True))
    .filter(F.col("code").isNotNull() & (F.trim(F.col("code")) != ""))
    .filter(F.col("activite").isNotNull() & (F.trim(F.col("activite")) != ""))
    .filter(F.col("user_id").isNotNull())
    .filter(F.col("startsAt_date").isNotNull())
    .filter(F.col("duration_hours") > 0)
    .withColumn(
        # code_site = code (5 chars) + activite (3 chars) = code_affaire (8 chars) côté Divalto
        "code_site_full",
        F.concat(F.trim(F.col("code")), F.trim(F.col("activite"))),
    )
    .withColumn(
        "external_id",
        F.md5(
            F.concat_ws(
                "|",
                F.coalesce(F.col("source_system"), F.lit("")),
                F.col("user_id").cast(StringType()),
                F.col("startsAt_date").cast(StringType()),
                F.coalesce(F.col("code"), F.lit("")),
                F.coalesce(F.col("activite"), F.lit("")),
            )
        ),
    )
)

nb_filtered = df_filtered.count()
print(f"[2] Après filtres (LUCCA + activite renseignée) : {nb_filtered} lignes")

In [ ]:
# ─────────────────────────────────────────
# 3) Mapping vers le schéma lucca_saisie_temps + résolution user_id
# ─────────────────────────────────────────

df_target = (
    df_filtered
    .select(
        F.col("external_id"),
        F.col("user_id").cast(LongType()).alias("id_lucca"),
        F.col("code_site_full").alias("code_site"),  # 8 chars (code + activite) = code_affaire
        F.col("startsAt_date").cast(DateType()).alias("date_saisie"),
        F.col("duration_hours").cast(DoubleType()).alias("duree_heures"),
        F.col("activite").cast(StringType()).alias("type_temps"),
        F.col("comment").cast(StringType()).alias("libelle"),
    )
)

rows = df_target.collect()
records = []
users_no_match = set()
for row in rows:
    d = row.asDict(recursive=True)
    id_lucca = d["id_lucca"]
    profile_id = ID_LUCCA_TO_PROFILE_ID.get(id_lucca) if id_lucca is not None else None
    if profile_id is None and id_lucca is not None:
        users_no_match.add(id_lucca)
    records.append({
        "external_id":        d["external_id"],
        "id_lucca":           id_lucca,
        "user_id":            profile_id,
        "code_site":          d["code_site"],
        "date_saisie":        d["date_saisie"],
        "duree_heures":       d["duree_heures"],
        "type_temps":         d["type_temps"],
        "libelle":            d["libelle"],
        "validated_by_lucca": False,
    })

print(f"[3] Records prêts : {len(records)}")
print(f"    Profils Lucca sans match Supabase : {len(users_no_match)}")
if users_no_match:
    print(f"    (ex. ids: {sorted(users_no_match)[:10]}{'...' if len(users_no_match) > 10 else ''})")

from collections import Counter
by_code = Counter(r["code_site"] for r in records)
print("    Top 10 codes_site (8 chars, format code_affaire) :")
for code, n in by_code.most_common(10):
    print(f"      {code} : {n} saisies")

In [ ]:
# ─────────────────────────────────────────
# 4) Envoi vers Supabase + résumé
# ─────────────────────────────────────────

ok, err = send_to_supabase(
    table        = "lucca_saisie_temps",
    conflict_key = "external_id",
    records      = records,
    label        = "LUCCA_TEMPS",
)

print("")
print("=" * 60)
print("SYNC LUCCA TEMPS → SUPABASE — RÉSUMÉ")
print("=" * 60)
print(f"  Source filter            : source_system = '{SOURCE_FILTER}'")
print(f"  Lignes filtrées Spark    : {nb_filtered}")
print(f"  Records prêts            : {len(records)}")
print(f"  Upsertés Supabase        : {ok}")
print(f"  Erreurs envoi            : {err}")
print(f"  user_id non résolu       : {len(users_no_match)} profils Lucca")
if err == 0:
    print("  ✅ Sync OK")
else:
    print("  ⚠️  Vérifier les logs ci-dessus")
print("=" * 60)